# Transformer Attention - From Scratch (NumPy)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/04_Deep_Learning/transformers/transformers_attention_from_scratch.ipynb)

The engine behind every modern LLM is **scaled dot-product attention**: each token builds a query (what am I looking for?), a key (what do I contain?) and a value (what do I pass along?). Attention scores = softmax(QK^T / sqrt(d)) V.

This notebook implements it in pure NumPy so every shape is visible. Runs anywhere - no GPU needed.

## 1. Softmax + scaled dot-product attention

In [ ]:
import numpy as np

np.set_printoptions(precision=2, suppress=True)

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)      # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)   # (batch, tgt, src)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    A = softmax(scores)
    return A @ V, A

np.random.seed(42)
B, T, D = 1, 4, 8                                # batch, seq len, embedding dim
X = np.random.randn(B, T, D)

W_q, W_k, W_v = (np.random.randn(D, D) * 0.3 for _ in range(3))
Q, K, V = X @ W_q, X @ W_k, X @ W_v

out, A = scaled_dot_product_attention(Q, K, V)
print("Q:", Q.shape, "-> attention matrix:", A.shape, "-> output:", out.shape)

Every output row is a **weighted average of all value vectors**, weights taken from row of A. Token 3 can look at tokens 0-2 - that is how context flows.

## 2. Visualize where each word looks

In [ ]:
import matplotlib.pyplot as plt

words = ["the", "cat", "sat", "mat"]
plt.imshow(A[0], cmap="viridis")
plt.xticks(range(T), words); plt.yticks(range(T), words)
plt.xlabel("Keys (attended to)"); plt.ylabel("Queries (from)")
plt.title("Attention weights"); plt.colorbar(label="weight")
plt.show()

## 3. Causal mask (decoder style)

In [ ]:
Tq = A.shape[1]
causal = np.tril(np.ones((Tq, Tq), dtype=bool))   # lower triangle = allowed
out_c, A_c = scaled_dot_product_attention(Q, K, V, mask=causal)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, M, t in zip(axes, [A[0], A_c[0]], ["bidirectional (encoder)", "causal (decoder)"]):
    ax.imshow(M, cmap="viridis"); ax.set_title(t)
    ax.set_xticks(range(T)); ax.set_xticklabels(words, rotation=45)
    ax.set_yticks(range(T)); ax.set_yticklabels(words)
plt.show()

The decoder triangle exists because during generation the model must not peek at future tokens.

## 4. Multi-head attention

In [ ]:
def multi_head_attention(X, n_heads):
    B, T, D = X.shape
    dh = D // n_heads
    Qh, Kh, Vh = X @ W_q, X @ W_k, X @ W_v          # project once
    # split into heads -> (B, h, T, dh)
    Qh, Kh, Vh = (a.reshape(B, T, n_heads, dh).transpose(0, 2, 1, 3)
                  for a in (Qh, Kh, Vh))
    scores = Qh @ Kh.transpose(0, 1, 3, 2) / np.sqrt(dh)
    A_h = softmax(scores)
    ctx = (A_h @ Vh).transpose(0, 2, 1, 3).reshape(B, T, D)
    return ctx, A_h

ctx, A_h = multi_head_attention(X, n_heads=2)
print("per-head attention:", A_h.shape, "-> concatenated context:", ctx.shape)

Each head learns a different relationship pattern (syntax, coreference, position...) before results are concatenated.

## 5. Sinusoidal positional encoding

In [ ]:
pos = np.arange(200)[:, None]
i   = np.arange(64)[None, :]
angle = pos / np.power(10000, 2 * i / 64)
pe = np.where(i % 2 == 0, np.sin(angle), np.cos(angle))

plt.figure(figsize=(9, 4))
plt.imshow(pe, aspect="auto", cmap="RdBu")
plt.xlabel("embedding dim"); plt.ylabel("token position")
plt.title("Sinusoidal positional encoding"); plt.colorbar(); plt.show()

## Key takeaways
- Attention = differentiable dictionary lookup (query matches keys, retrieves values).
- Complexity O(T^2 d) - why context length is expensive.
- A full transformer block = MultiHeadAttention + FFN + residual connections + LayerNorm.
- Next step: see these blocks pre-trained at scale in `huggingface_transformers_basics.ipynb`.